### Смоделируем реальную задачу: высокочастотный датчик вибрации на роторе турбины.

В таких данных простые средние значения (rolling.mean) или IsolationForest бессильны. Нормальная работа турбины — это не плоская линия, это синусоида (колесо крутится, вибрация идет волной).

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

np.random.seed(42)

# 1. Генериурем сигнал 5000 точек, от 0.0 до 50.0 секунд, Шаг (~0.01 сек)
# Частота дискретизации — равна 100 Гц (100 измерений в секунду).
t = np.linspace(0, 50, 5000)

# Основная синусоида (идеального круга не существует)
# Один полный цикл синусоиды (от пика до пика) — это ровно один оборот вала турбины.
rotation = np.sin(2 * np.pi * 1 * t)

# Шум (электромагнитные помехи, трение, потоки пара/газа)
# Нормальное распределение (или распределение Гаусса) - симметричный колокол
noise = np.random.normal(0, 0.2, 5000)

# Сигналы вибрации
vibration = rotation + noise

# моделируется работа системы диагностики турбины
y_true = np.zeros(5000) # 0 - норма, в эту мс турбина работает исправно

# Ложная аномалия (шум связи, случ. удар) - точка 1500
vibration[1500] += 5.0

# Реальная поломка: последние 500 точек появляется высокочастотная вибрация
vibration[-500:] += 0.8 * np.sin(10 * np.pi * t[-500:])
y_true[-500:] = 1

df = pd.DataFrame({"Vibration": vibration, "True_Label": y_true})

df.head()

,Vibration,True_Label
0,0.099343,0.0
1,0.035150,0.0
2,0.254896,0.0
3,0.492024,0.0
4,0.201908,0.0


#### 2. Подготовка окон (window_size = 50, чтобы захватить период синусоиды)

**StandardScaler** — делает масштаб «похожим на колокол» распределения Гаусса. Центр равен 0, базовый шаг равен 1, а аномалии могут принимать любые гигантские значения (10, 30, 100), выделяясь на общем фоне.

In [ ]:
window_size = 50

# Вызываем конструктор класса и создаем в оперативной памяти конкретный «физический» экземпляр (выделенную структуру данных)
# Методы: .fit(), .transform(), .fit_transform()
# Атрибуты: mean_, scale_, var_
scaler = StandardScaler()

# scaler.mean_ — сюда записывается массив с вычисленным средним значением
# scaler.scale_ — сюда записывается массив со стандартным отклонением (разбросом).
# scaler.n_samples_seen_ — количество строк, которые он прочитал (2000).
scaler.fit(df[["Vibration"]].iloc[:2000])

# z = (x - mean_ / scale_) - новое отмасштабированное значение
# Стандартизированная копия df["Vibration"]
scaled_vibration = scaler.transform(df[["Vibration"]])
print(type(scaled_vibration))
print(scaled_vibration.shape) # Колонка: 1 (Вибрация), Строки: 5000 (Значения)
print(scaled_vibration[:5])
print(scaled_vibration[1500])

<class 'numpy.ndarray'>
(5000, 1)
[[0.11811719]
 [0.03178823]
 [0.32731152]
 [0.64621188]
 [0.25605105]]
[6.94343617]
